# Module 04 · Sampling and Monte Carlo CFR

Compare sampling schemes on Kuhn, where exact values and exploitability are cheap enough to calculate. Run from anywhere inside the repository.

In [ ]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / 'pokerbot').is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError('Run this notebook inside the pokerbots repository')

## Exact answer for a reference strategy

The α = 0 member of the Kuhn equilibrium family is a known anchor. Use it to check the value estimators.

In [ ]:
from pokerbot.games.kuhn import KuhnPoker
from pokerbot.solvers.mccfr import MCCFRTrainer, estimate_value
from pokerbot.eval.exploitability import expected_value, exploitability

game = KuhnPoker()
nash = {
    'J': [1, 0], 'Q': [1, 0], 'K': [1, 0],
    'Jpb': [1, 0], 'Qpb': [2/3, 1/3], 'Kpb': [0, 1],
    'Jp': [2/3, 1/3], 'Qp': [1, 0], 'Kp': [0, 1],
    'Jb': [1, 0], 'Qb': [2/3, 1/3], 'Kb': [0, 1],
}
exact_value = expected_value(game, nash, 0)
print(f'exact value: {exact_value:.6f}  (known answer: {-1/18:.6f})')
print(f'exact exploitability: {exploitability(game, nash):.8f}')
assert abs(exact_value + 1/18) < 1e-10

## Sample the fixed profile

Chance sampling draws a deal but integrates all actions. Outcome sampling draws both chance and decisions. Increase the sample count or change the seed; individual estimates move, while the mean remains near the exact value.

In [ ]:
samples = 4000
for scheme in ('chance', 'outcome'):
    estimate = estimate_value(game, nash, samples=samples, sampling=scheme, seed=7)
    print(f'{scheme:7s}: estimate={estimate:+.5f}, error={estimate-exact_value:+.5f}')
    assert abs(estimate - exact_value) < 0.06

## Train sampled CFR

The trainer reports strategies, not an exploitability estimate. On Kuhn we can evaluate each average strategy exactly. Fixed seeds make the comparison reproducible; runtime includes Python traversal overhead.

In [ ]:
checkpoints = (100, 500, 2000, 10000)
for scheme in ('chance', 'external'):
    trainer = MCCFRTrainer(game, sampling=scheme, seed=4)
    previous = 0
    print(f'\n{scheme} sampling')
    for checkpoint in checkpoints:
        trainer.train(checkpoint - previous)
        previous = checkpoint
        average = trainer.average_strategy()
        print(f'{checkpoint:>6} iterations  exploitability={exploitability(game, average):.5f}  value={expected_value(game, average):+.5f}')

## Predict → play → prove → probe

Before training, predict which sampler will make more node visits per iteration. Run both and compare exploitability against iteration count. Then rerun with another seed and explain what changed. A smaller per-iteration tree does not guarantee a smoother curve.